# Verification of the modulo-81 identities in Sage

Following the modulo-256 playground, we define the relations, load the precomputed source modules and recorded intermediate elements, and check the defining equations on every cyclic generator.

All arithmetic replay is performed in Sage. No Nim process, search for intermediate elements, or recursive verification is used. The source presentations and Hecke matrices are taken as inputs; their construction and descent are not checked here.

This verifies the **finite-source conditions** in the manuscript's prime-3 section. Propagation, passage to the free target lattice, the eigenvalue deductions, and strong realization are separate arguments.

| Group | Working modulus | Even coefficient degrees | Source |
| :--- | :--- | :--- | :--- |
| Common $T_7$ | $81$ | $0\le d<270$ | $\mathbb M_d^{(-1)^q}$ |
| Nonzero branch | $243$ | $0\le d<810$, $d\bmod6\in\{0,4\}$ | $\mathbb M_d^{(-1)^q}$ |
| Zero branch | $2187$ | $0\le d<7290$, $d\bmod6=2$ | $I\mathbb M_d^{(-1)^q}$, $I=(9,t_2)$ |

Both orientations $q=0,1$ are checked, in ascending degree order. These are 270, 540, and 2430 cases respectively. No transfer-map shortcuts are used.


In [3]:
import gzip
import hashlib
import json
import sys
from pathlib import Path
from sage.all import *

ROOT = Path.cwd()
if not (ROOT / 'python').is_dir():
    raise FileNotFoundError('Open this notebook from the repository root.')
sys.path.insert(0, str(ROOT / 'python'))
from load_source_data import load_source_data, load_ideal_source_data
from verify_hecke_relations import relation_spec

p = 3
relation_period = 54
VERIFICATION_DIRECTORY = ROOT / 'verification_data/mod81_compact'


## 1. Define all the relations

Throughout, $r=d\bmod54$ and $\epsilon_q=(-1)^q$. The operators are oriented **once**, by $t_n=n^{q3^{m-1}}T_n$ at working modulus $3^m$.

### Common $T_7$ relation

Let $t_r$ be the representative of $1+7^{r+1}\bmod81$ in $\{0,\ldots,80\}$. On the signed source modulo 81, put

$$
\mathcal N_{r,q}=\mathfrak D_{t_7-t_r,27},\qquad
\operatorname{dom}\mathcal N_{r,q}=M,\qquad
\mathcal N_{r,q}^2\equiv0\pmod3.
$$

Here $t_7=T_7$ in both orientations.

### Nonzero branch

Use the manuscript's constants $c_r$ below. On the signed source modulo 243, put

$$
\mathcal Q_{r,q}=\mathfrak D_{t_2^2-9c_r,81},\qquad
F(X)=X^4-2X^3+X^2.
$$

Check full domain and $F(\mathcal Q_{r,q})\equiv0\pmod3$.

### Zero branch

All intermediate elements belong to the **ideal image** $M=I\mathbb M_d^{\epsilon_q}(\mathbb Z/2187\mathbb Z)$, not the whole Manin quotient. Put

$$
G_r(X)=(X^3-X)^2+3h_r(X)(X^3-X),
$$

$$
\mathcal A_q=\mathfrak D_{t_2,9}(M),\qquad
\mathcal B_{r,q}=\mathfrak D_{1,9}\circ G_r(\mathcal A_q).
$$

With the displayed lifts of $h_r$ modulo 9 and the sets $E_{r,s}$ below, check

$$
F_0(\mathcal B_{r,q})\equiv0\pmod3,\qquad
F_{r,s}(\mathcal A_q,\mathcal B_{r,q})\equiv0\pmod3
\quad(s=0,1,2),
$$

where

$$
F_0(Y)=(Y^3-Y)^2,
$$

$$
F_{r,s}(X,Y)=(1-(X-s)^2)^6
\prod_{\delta\in E_{r,s}}(Y-\delta)^2.
$$

Polynomials are expanded before evaluation. In each monomial $X^iY^j$, apply $\mathcal B_{r,q}$ first, then $\mathcal A_q$. The source symbols $N,Q,A,B$ in the code stand for these linear relations, not globally divided source endomorphisms.


In [2]:
# Common T7 relation, modulo 81.
R_T7 = Integers(3^4)
S_T7.<T7,N> = PolynomialRing(R_T7)
t_r = {r: (1+7^(r+1)) % 81 for r in range(0,54,2)}
spec_T7 = {
    r: relation_spec(
        [('N_full_domain', N, 0), ('N_terminal', N^2, 1)],
        hecke_operators={'T7': 7},
        divisions={'N': (T7-t_r[r], 3)},
        witness_semantics='independent_monomials',
        max_howell_dimension=4096,
    ) for r in t_r
}

# Nonzero branch, modulo 243: manuscript c_r and F.
R_Q = Integers(3^5)
S_Q.<T2,Q> = PolynomialRing(R_Q)
c_r = {
    0: 1, 4: 4, 6: 4, 10: 1, 12: 7, 16: -2,
    18: 10, 22: -5, 24: 13, 28: -8, 30: 16, 34: 16,
    36: -8, 40: 13, 42: -5, 46: 10, 48: -2, 52: 7,
}
F = Q^4-2*Q^3+Q^2
spec_Q = {
    r: relation_spec(
        [('Q_full_domain', Q, 0), ('Q_terminal', F, 1)],
        hecke_operators={'T2': 2},
        divisions={'Q': (T2^2-9*c_r[r], 4)},
        witness_semantics='independent_monomials',
        max_howell_dimension=4096,
    ) for r in c_r
}

# Zero branch, modulo 2187: manuscript h_r, E_{r,s}, G_r, F_0, F_{r,s}.
R_I = Integers(3^7)
S_I.<T,A,B> = PolynomialRing(R_I)
h_r, E_rs = {}, {}
for r in (2,20,38):
    h_r[r] = 1-A^2
    E_rs[r] = ((0,2),(0,1),(0,1))
for r in (8,26,44):
    h_r[r] = 1-A^2
    E_rs[r] = ((0,2),(1,2),(1,2))
h_r[14] = 2+5*A
E_rs[14] = ((0,2),(0,2),(0,1))
for r in (32,50):
    h_r[r] = 2+A^2
    E_rs[r] = ((0,2),(0,1),(0,1))
G_r = {r: (A^3-A)^2+3*h_r[r]*(A^3-A) for r in h_r}
F_0 = (B^3-B)^2
F_rs = {
    (r,s): (1-(A-s)^2)^6 * prod((B-delta)^2 for delta in E_rs[r][s])
    for r in h_r for s in range(3)
}
spec_I = {
    r: relation_spec(
        [('Frobenius', F_0, 1)] +
        [(f'branch_{s}', F_rs[r,s], 1) for s in range(3)],
        hecke_operators={'T': 2},
        divisions={'A': (T, 2), 'B': (G_r[r], 2)},
        witness_semantics='independent_monomials',
        max_howell_dimension=4096,
    ) for r in h_r
}

groups = {
    'T7_mod81': {
        'ring': R_T7, 'exponent': 4, 'scope': 'manin',
        'sources': ROOT / 'source_data/p3_T7_mod81',
        'degrees': tuple(range(0,270,2)), 'specifications': spec_T7,
    },
    'nonzero_mod243': {
        'ring': R_Q, 'exponent': 5, 'scope': 'manin',
        'sources': ROOT / 'source_data/p3_nonzero_T2_T7_mod243',
        'degrees': tuple(d for d in range(0,810,2) if d%6 in (0,4)),
        'specifications': spec_Q,
    },
    'zero_ideal_mod2187': {
        'ring': R_I, 'exponent': 7, 'scope': 'ideal_image',
        'sources': ROOT / 'source_data/p3_ideal_9_T2_mod2187',
        'degrees': tuple(d for d in range(0,7290,2) if d%6 == 2),
        'specifications': spec_I,
    },
}
# These comparisons bind the visible definitions to the saved data.
for name, group in groups.items():
    plan = json.loads((ROOT / f'relations/p3_mod81_{name}_native.json').read_text())
    assert {str(r): spec for r, spec in group['specifications'].items()} == plan['relation_specifications']
    assert group['ring'].characteristic() == p^plan['exponent']
    expected = tuple(d for d in range(0,plan['degree_bound'],2)
                     if d % plan['degree_residue_modulus'] in plan['degree_residues'])
    assert group['degrees'] == expected
    print(name, 'modulus:', group['ring'].characteristic(),
          '| degrees:', len(group['degrees']), '| cases:', 2*len(group['degrees']))


T7_mod81 modulus: 81 | degrees: 135 | cases: 270
nonzero_mod243 modulus: 243 | degrees: 270 | cases: 540
zero_ideal_mod2187 modulus: 2187 | degrees: 1215 | cases: 2430


## 2. Recover intermediate elements and check the equations

For an equation $p^\alpha y=u$, the recorded data in these files use the least nonnegative coordinate representative of $u$, divided by $p^\alpha$, with no further kernel corrections. The equation is then checked in the source, coordinate by coordinate. This is an elementwise choice, not an endomorphism.

For nested relations, the helper lists the equations obtained from the expanded polynomials. It applies the rightmost relation first and shares intermediate elements where the recorded construction uses a common chain. Every equation, including the final $y=p^b\rho$, is checked.

The archive calls each record a “packet”. Its “binding” is an identifier for the source, oriented matrices, coordinates, generators, and presentations. Matching that identifier is only a file-consistency check; it does not establish the equations.

The file-format names containing “witness” are retained for compatibility and refer to the intermediate elements of these presentations.

To bound memory, the arithmetic is performed on small batches of cyclic generators. This is sequential processing, not parallel computation.


In [3]:
GENERATOR_BATCH_SIZE = 4

def compact_json(value):
    """
    Serialize the source data and presentation in the archive's JSON format.
    Return the text used to identify the corresponding verification data.
    Values must already be JSON-compatible, including Python integers.
    """
    return json.dumps(value, separators=(',', ':'), ensure_ascii=False)

def nim_sequence(values):
    """
    Encode a sequence of integers in the archive's '@[1, 2, ...]' format.
    This is used only for the identifier of the recorded data; no Nim
    computation is performed.
    """
    return '@[' + ', '.join(str(int(v)) for v in values) + ']'

def packet_binding(spec, metadata, moduli, operators, exponent):
    """
    Identify the recorded data for this source and these presentations.
    
    The archive calls this identifier a 'binding'. It is computed from
    the coefficient degree, modulus, source metadata, cyclic coordinates,
    oriented Hecke matrices, tested generators, and the specified relations.
    
    Changing the degree, coordinates, or presentation generally requires
    different intermediate elements. Reproducing the stored identifier
    ensures that the data correspond to the inputs under consideration.
    
    This file-consistency check does not establish a mathematical identity.
    The defining equations of the presentations are checked separately
    by replay_equations. Return the identifier in the archive's format.
    The exponent specifies the group's working precision.
    """
    rank = len(moduli)
    parts = [compact_json(spec), f'{p}:{exponent}', compact_json(metadata),
             nim_sequence(moduli),
             f'{rank}:{rank}:' + nim_sequence(
                 1 if i == j else 0 for i in range(rank) for j in range(rank))]
    for name in spec['variables']:
        if name in operators:
            parts.append(name + ':' + nim_sequence(operators[name].list()))
    return hashlib.sha1(''.join(parts).encode()).hexdigest().upper()

def load_record(directory, binding, name, metadata, rank, modulus):
    """
    Load the recorded intermediate-element data for one relation.
    
    Check the source identifier, relation name, working modulus, and
    number of cyclic generators. Return the decoded archive record.
    
    The saved data specify how to choose the intermediate elements
    x_1, x_2, ... in the defining equations. For p^alpha*y = u,
    reduce each coordinate of u to its least nonnegative representative
    and divide by p^alpha, provided the required divisibility holds.
    The resulting equation is checked separately in the source module.
    
    The present files use these choices without further adjustments.
    If a file specifies different choices, this verifier stops because
    it does not implement their reconstruction. No search for
    intermediate elements is performed. These elementwise choices
    need not define an endomorphism of the source.
    """
    suffix = hashlib.sha1(name.encode()).hexdigest().upper()
    path = directory / f'{binding}_{suffix}.json.gz'
    with gzip.open(path, 'rt') as stream:
        record = json.load(stream)
    assert record['schema'] in ('hecke.compact-relation-witness.v1',
                                'hecke.compact-relation-witness.v2'), path
    assert record['binding'] == binding and record['relation'] == name, path
    assert record['source'] == metadata, path
    assert record['working_modulus'] == modulus and record['input_count'] == rank, path
    if record['independent'] is not False or record['choices'] != [] or 'recipe' in record:
        raise NotImplementedError(f'{path}: unsupported recorded intermediate elements')
    return record

def mixed_zero(value, moduli):
    """
    Check equality to zero in the source module in cyclic coordinates.
    
    The rows represent elements of M = direct_sum_j Z/moduli[j]Z.
    Each column is reduced modulo its own cyclic order, rather than the
    ambient modulus. Return True precisely when all rows represent zero.
    For the zero module, this condition holds vacuously.
    """
    return all(ZZ(value[i,j]) % order == 0 for i in range(value.nrows())
               for j, order in enumerate(moduli))

def canonical_preimage(rhs, divisor, moduli):
    """
    Recover the recorded y in the relation rhs = divisor*y on M.
    
    The rows of rhs are elements of M in cyclic coordinates. The divisor
    and cyclic orders are powers of the same prime. In each coordinate,
    take the least nonnegative residue of rhs and divide it by divisor,
    using the coordinatewise choice specified by the recorded data.
    
    Return the rows y after checking divisor*y = rhs in M. Failure of
    the required divisibility raises ArithmeticError. These are choices
    of intermediate elements, not a globally defined divided endomorphism.
    """
    output = zero_matrix(rhs.base_ring(), rhs.nrows(), rhs.ncols())
    for i in range(rhs.nrows()):
        for j, order in enumerate(moduli):
            entry = ZZ(rhs[i,j]) % order
            if entry % gcd(divisor, order):
                raise ArithmeticError(f'Division equation fails in coordinate ({i},{j})')
            output[i,j] = entry // divisor
    assert mixed_zero(divisor*output-rhs, moduli)
    return output

def defining_equations(spec, relation):
    """
    List the defining equations of the specified presented relation.
    
    The inputs are the specification and one polynomial congruence
    condition. Each numbered output represents an intermediate element;
    the initial element has number zero.
    
    Expand all polynomials, including numerators in nested divisions.
    In each monomial, apply the rightmost relation first. Reuse intermediate
    elements where the recorded construction uses a common chain. This
    supplies an element of the polynomial relation; it does not simplify
    products of multivalued relations.
    
    Append the equation y = p^b*rho, expressing membership in p^b M.
    Return the list of equations; replay_equations checks them in M.
    """
    variables = spec['variables']
    definitions = {item['variable']: item for item in spec['divisions']}
    ordinary = set(spec['hecke_operators'])
    assert not (ordinary & set(definitions))
    assert ordinary | set(definitions) == set(variables)
    assert not spec.get('presentations')
    assert 'input_polynomial' not in relation and 'input_presentation' not in relation
    for name, definition in definitions.items():
        index = variables.index(name)
        assert all(all(e == 0 for e in powers[index:])
                   for coefficient, powers in definition['numerator'])
    steps, cache = [], {}

    def polynomial(terms, input_node):
        """
        List the outputs contributing to the polynomial at input_node.
        
        The terms are pairs [coefficient, exponent list]. Apply relations
        in the displayed order, with the rightmost first. Return pairs
        consisting of an output number and its coefficient.
        
        These numbers identify intermediate elements to be recovered later.
        No commutation or algebraic simplification of divided relations
        is used.
        """
        result = []
        for coefficient, powers in terms:
            assert len(powers) == len(variables) and all(e >= 0 for e in powers)
            if ZZ(coefficient) % spec['coefficient_modulus'] == 0:
                continue
            node = input_node
            for name, power in reversed(list(zip(variables, powers))):
                for _ in range(power):
                    node = apply_relation(name, node)
            result.append((node, ZZ(coefficient)))
        return result

    def apply_relation(name, input_node):
        """
        Add the equations for applying one relation to a specified input.
        
        For an ordinary operator, record its action. For a division relation,
        expand the numerator first and then record the division equation.
        Return the number of the resulting intermediate element.
        
        Repeated applications to the same numbered input share that output.
        This implements the recorded common-chain construction, not a
        globally defined divided endomorphism.
        """
        key = (name, input_node)
        if key in cache:
            return cache[key]
        if name in ordinary:
            step = ('ordinary', name, input_node)
        else:
            definition = definitions[name]
            terms = polynomial(definition['numerator'], input_node)
            step = ('division', p^definition['power'], terms)
        steps.append(step)
        node = len(steps)
        cache[key] = node
        return node

    terminal = polynomial(relation['polynomial'], 0)
    steps.append(('division', p^relation['terminal_power'], terminal))
    return steps

def replay_equations(steps, operators, moduli, ring):
    """
    Verify the defining equations on every cyclic generator of M.
    
    The inputs are the list of equations, oriented Hecke matrices,
    cyclic coordinate orders, and the working coefficient ring.
    Recover the recorded intermediate elements and check each equation,
    including those introduced in nested divisions.
    
    The final equation y = p^b*rho places the chosen polynomial output
    in p^b M. Checking all cyclic generators suffices, since the
    corresponding presented relation is linear.
    
    Process a few generators at a time and discard intermediate matrices
    after their last use. Every equation is still checked.
    Return None on success; failed checks stop the verification.
    This establishes finite-source conditions only, not propagation or
    the divided endomorphisms on the free target lattice.
    """
    rank = len(moduli)
    uses = [0] * (len(steps)+1)
    for kind, operation, inputs in steps:
        dependencies = [inputs] if kind == 'ordinary' else [node for node, c in inputs]
        for node in dependencies:
            uses[node] += 1
    for start in range(0, rank, GENERATOR_BATCH_SIZE):
        count = min(GENERATOR_BATCH_SIZE, rank-start)
        initial = zero_matrix(ring, count, rank)
        for i in range(count):
            initial[i,start+i] = 1
        values = {0: initial}
        remaining = list(uses)
        for number, (kind, operation, inputs) in enumerate(steps, start=1):
            if kind == 'ordinary':
                output = values[inputs] * operators[operation]
                dependencies = [inputs]
            else:
                rhs = zero_matrix(ring, count, rank)
                for node, coefficient in inputs:
                    rhs += coefficient * values[node]
                output = canonical_preimage(rhs, operation, moduli)
                dependencies = [node for node, c in inputs]
            values[number] = output
            for node in dependencies:
                remaining[node] -= 1
                if remaining[node] == 0:
                    del values[node]
            if remaining[number] == 0:
                del values[number]

def replay_mod81_case(group_name, d, q):
    """
    Verify the source identities in coefficient degree d and orientation q.
    
    Select the common T7, nonzero, or zero-branch group. Load the signed
    modular-symbol source, or its ideal image for the zero branch, in
    cyclic coordinates. All intermediate elements belong to this same
    loaded module.
    
    The archived Hecke matrices are untwisted. Apply n^(q*3^(m-1))
    exactly once, with m the group's working exponent. Check compatibility
    with the cyclic orders and identify the recorded data for the
    specified presentations. Then check their equations in Sage.
    
    Return the group, degree, orientation, working modulus, number of
    cyclic coordinates, number of archive records read, and success.
    The Manin presentation and Hecke actions are inputs: they are not
    reconstructed or checked for descent here. All-weight propagation
    and passage to the target lattices are separate arguments.
    """
    group = groups[group_name]
    assert q in (0,1) and d in group['degrees']
    ring, exponent = group['ring'], group['exponent']
    modulus = ring.characteristic()
    spec = group['specifications'][d % relation_period]
    path = (group['sources'] / f'degree_{d}.npz').resolve()
    if group['scope'] == 'ideal_image':
        data = load_ideal_source_data(ring, d, q, path)
        assert data['source_scope'].startswith('ideal_image')
        # Metadata convention of this legacy (9,T2) ideal-image archive.
        metadata = json.loads('{"source_scope":"ideal_image","ideal":{"scalar":9,"generators":[[[1,[1]]]]}}')
    else:
        data = load_source_data(ring, d, q, path)
        assert data['source_scope'] == 'manin'
        metadata = dict(data['source_metadata'])
    moduli = [int(order) for order in data['coordinates']['coordinate_moduli']]
    rank = len(moduli)
    assert all(order > 1 and modulus % order == 0 for order in moduli)
    operators = {}
    for name, n in spec['hecke_operators'].items():
        archived = data['archived_hecke_matrices'][n]['normalized_matrix']
        twist = power_mod(n, q*p^(exponent-1), modulus)
        action = matrix(ring, rank, rank,
                        [(ZZ(archived[i,j])*twist) % moduli[j]
                         for i in range(rank) for j in range(rank)])
        assert all(moduli[i]*ZZ(action[i,j]) % moduli[j] == 0
                   for i in range(rank) for j in range(rank))
        operators[name] = action
    metadata.update(degree=int(d), orientation=int(q), sign=int((-1)^q),
                    source_path=str(path), source_sha256=data['source_sha256'])
    binding = packet_binding(spec, metadata, moduli, operators, exponent)
    directory = VERIFICATION_DIRECTORY / group_name / 'packets'
    for relation in spec['relations']:
        load_record(directory, binding, relation['name'], metadata, rank, modulus)
        steps = defining_equations(spec, relation)
        replay_equations(steps, operators, moduli, ring)
    return {'group': group_name, 'degree': d, 'orientation': q,
            'sign': (-1)^q, 'working_modulus': modulus, 'rank': rank,
            'packets_loaded': len(spec['relations']), 'passed': True}


## Common T7 relation

Run this loop after the definitions and helpers. It checks both orientations in ascending degree order. For a short trial, replace the degree list in the loop by a few valid degrees; the printed count then refers only to those cases.


In [4]:
results_T7_mod81 = []
for d in groups['T7_mod81']['degrees']:
    for q in (0,1):
        test = replay_mod81_case('T7_mod81', d, q)
        results_T7_mod81.append(test)
        print(test, flush=True)
print('Cases replayed:', len(results_T7_mod81))
print('ALL REQUESTED T7_mod81 FINITE-SOURCE CONDITIONS VERIFIED IN SAGE')


{'group': 'T7_mod81', 'degree': 0, 'orientation': 0, 'sign': 1, 'working_modulus': 81, 'rank': 0, 'packets_loaded': 2, 'passed': True}


{'group': 'T7_mod81', 'degree': 0, 'orientation': 1, 'sign': -1, 'working_modulus': 81, 'rank': 0, 'packets_loaded': 2, 'passed': True}
{'group': 'T7_mod81', 'degree': 2, 'orientation': 0, 'sign': 1, 'working_modulus': 81, 'rank': 1, 'packets_loaded': 2, 'passed': True}
{'group': 'T7_mod81', 'degree': 2, 'orientation': 1, 'sign': -1, 'working_modulus': 81, 'rank': 0, 'packets_loaded': 2, 'passed': True}
{'group': 'T7_mod81', 'degree': 4, 'orientation': 0, 'sign': 1, 'working_modulus': 81, 'rank': 1, 'packets_loaded': 2, 'passed': True}
{'group': 'T7_mod81', 'degree': 4, 'orientation': 1, 'sign': -1, 'working_modulus': 81, 'rank': 0, 'packets_loaded': 2, 'passed': True}
{'group': 'T7_mod81', 'degree': 6, 'orientation': 0, 'sign': 1, 'working_modulus': 81, 'rank': 1, 'packets_loaded': 2, 'passed': True}
{'group': 'T7_mod81', 'degree': 6, 'orientation': 1, 'sign': -1, 'working_modulus': 81, 'rank': 1, 'packets_loaded': 2, 'passed': True}
{'group': 'T7_mod81', 'degree': 8, 'orientation': 0

## Nonzero branch

Run this loop after the definitions and helpers. It checks both orientations in ascending degree order. For a short trial, replace the degree list in the loop by a few valid degrees; the printed count then refers only to those cases.


In [5]:
results_nonzero_mod243 = []
for d in groups['nonzero_mod243']['degrees']:
    for q in (0,1):
        test = replay_mod81_case('nonzero_mod243', d, q)
        results_nonzero_mod243.append(test)
        print(test, flush=True)
print('Cases replayed:', len(results_nonzero_mod243))
print('ALL REQUESTED nonzero_mod243 FINITE-SOURCE CONDITIONS VERIFIED IN SAGE')


{'group': 'nonzero_mod243', 'degree': 0, 'orientation': 0, 'sign': 1, 'working_modulus': 243, 'rank': 0, 'packets_loaded': 2, 'passed': True}
{'group': 'nonzero_mod243', 'degree': 0, 'orientation': 1, 'sign': -1, 'working_modulus': 243, 'rank': 0, 'packets_loaded': 2, 'passed': True}
{'group': 'nonzero_mod243', 'degree': 4, 'orientation': 0, 'sign': 1, 'working_modulus': 243, 'rank': 1, 'packets_loaded': 2, 'passed': True}
{'group': 'nonzero_mod243', 'degree': 4, 'orientation': 1, 'sign': -1, 'working_modulus': 243, 'rank': 0, 'packets_loaded': 2, 'passed': True}
{'group': 'nonzero_mod243', 'degree': 6, 'orientation': 0, 'sign': 1, 'working_modulus': 243, 'rank': 1, 'packets_loaded': 2, 'passed': True}
{'group': 'nonzero_mod243', 'degree': 6, 'orientation': 1, 'sign': -1, 'working_modulus': 243, 'rank': 1, 'packets_loaded': 2, 'passed': True}
{'group': 'nonzero_mod243', 'degree': 10, 'orientation': 0, 'sign': 1, 'working_modulus': 243, 'rank': 2, 'packets_loaded': 2, 'passed': True}
{'

## Zero branch on the ideal images

Run this loop after the definitions and helpers. It checks both orientations in ascending degree order. For a short trial, replace the degree list in the loop by a few valid degrees; the printed count then refers only to those cases.


In [6]:
results_zero_ideal_mod2187 = []
for d in groups['zero_ideal_mod2187']['degrees']:
    for q in (0,1):
        test = replay_mod81_case('zero_ideal_mod2187', d, q)
        results_zero_ideal_mod2187.append(test)
        print(test, flush=True)
print('Cases replayed:', len(results_zero_ideal_mod2187))
print('ALL REQUESTED zero_ideal_mod2187 FINITE-SOURCE CONDITIONS VERIFIED IN SAGE')


{'group': 'zero_ideal_mod2187', 'degree': 2, 'orientation': 0, 'sign': 1, 'working_modulus': 2187, 'rank': 1, 'packets_loaded': 4, 'passed': True}
{'group': 'zero_ideal_mod2187', 'degree': 2, 'orientation': 1, 'sign': -1, 'working_modulus': 2187, 'rank': 0, 'packets_loaded': 4, 'passed': True}
{'group': 'zero_ideal_mod2187', 'degree': 8, 'orientation': 0, 'sign': 1, 'working_modulus': 2187, 'rank': 1, 'packets_loaded': 4, 'passed': True}
{'group': 'zero_ideal_mod2187', 'degree': 8, 'orientation': 1, 'sign': -1, 'working_modulus': 2187, 'rank': 0, 'packets_loaded': 4, 'passed': True}
{'group': 'zero_ideal_mod2187', 'degree': 14, 'orientation': 0, 'sign': 1, 'working_modulus': 2187, 'rank': 2, 'packets_loaded': 4, 'passed': True}
{'group': 'zero_ideal_mod2187', 'degree': 14, 'orientation': 1, 'sign': -1, 'working_modulus': 2187, 'rank': 1, 'packets_loaded': 4, 'passed': True}
{'group': 'zero_ideal_mod2187', 'degree': 20, 'orientation': 0, 'sign': 1, 'working_modulus': 2187, 'rank': 2, 'p

KeyboardInterrupt: 

## Optional: replay the same recorded data with Nim

This is an alternative to the Sage loops above. Run the imports and relation-definitions cells first; the Sage verification loops do not need to have completed.

Set `RUN_NIM_REPLAY = True` to check all three groups with `NIM_WORKERS = 4` by default. Change `NIM_WORKERS` to choose another positive worker count. The native verifier reads the same source data and recorded intermediate elements, checks all defining equations, and stops if any required record is missing. It does not search for new intermediate elements or use recursive verification.

For a short trial, leave the flag false and call, for example, `replay_mod81_case_nim('T7_mod81', 12, 0)` after running this cell. The existing compiled verifier must be available; this cell does not rebuild it.

Cases are submitted in ascending degree order within each group, in bounded batches. Completion messages may appear out of order. Each worker runs one case; processes are released after each batch. If a case fails, the current batch finishes but no further batch is submitted. Parallel replay does not require recursion or earlier-degree results.


In [7]:
import os
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context

RUN_NIM_REPLAY = True
NIM_WORKERS = 4
NIM_VERIFIER = ROOT / 'nim/.verify-hecke-relations-build/verify_hecke_relations'

def replay_mod81_case_nim(group_name, d, q):
    """
    Verify the same finite-source conditions using the optional Nim replay.

    Load the signed source or ideal image and orient its Hecke matrices
    exactly as in the Sage verification. Pass these inputs, the relation
    specification, and the recorded-data directory to the native verifier.

    Require full-source verification and reuse of every saved record.
    Missing data or failed defining equations stop the call. The request
    uses replay mode only: no search, recursive shortcuts, or persistent
    checkpoints are requested. Return the complete native report.

    This is an alternative to the Sage equation checks, not a prerequisite
    for them. Source construction and all-weight propagation remain outside
    the scope of both replays.
    """
    group = groups[group_name]
    assert q in (0,1) and d in group['degrees']
    ring, exponent = group['ring'], group['exponent']
    modulus = ring.characteristic()
    spec = group['specifications'][d % relation_period]
    path = (group['sources'] / f'degree_{d}.npz').resolve()
    if group['scope'] == 'ideal_image':
        data = load_ideal_source_data(ring, d, q, path)
        assert data['source_scope'].startswith('ideal_image')
        metadata = json.loads('{"source_scope":"ideal_image","ideal":{"scalar":9,"generators":[[[1,[1]]]]}}')
    else:
        data = load_source_data(ring, d, q, path)
        assert data['source_scope'] == 'manin'
        metadata = dict(data['source_metadata'])
    moduli = [int(order) for order in data['coordinates']['coordinate_moduli']]
    rank = len(moduli)
    operators = {}
    for name, n in spec['hecke_operators'].items():
        archived = data['archived_hecke_matrices'][n]['normalized_matrix']
        twist = power_mod(n, q*p^(exponent-1), modulus)
        operators[name] = [
            [int((ZZ(archived[i,j])*twist) % moduli[j]) for j in range(rank)]
            for i in range(rank)
        ]
    metadata.update(degree=int(d), orientation=int(q), sign=int((-1)^q),
                    source_path=str(path), source_sha256=data['source_sha256'])
    request = {
        'source': {
            'schema': 'hecke.mixed-source.v1',
            'prime': int(p), 'exponent': int(exponent),
            'coordinate_moduli': moduli, 'operators': operators,
            'operators_are_oriented': True, 'metadata': metadata,
        },
        'relations': spec,
        'witness_directory': str(VERIFICATION_DIRECTORY / group_name / 'packets'),
        'witness_mode': 'replay',
    }
    if not NIM_VERIFIER.is_file():
        raise FileNotFoundError(f'Native verifier not found: {NIM_VERIFIER}')
    environment = os.environ.copy()
    environment['LD_LIBRARY_PATH'] = (
        str(Path(sys.prefix) / 'lib') + os.pathsep +
        environment.get('LD_LIBRARY_PATH', '')
    )
    process = subprocess.run(
        [str(NIM_VERIFIER), '-'], input=json.dumps(request),
        text=True, capture_output=True, env=environment,
    )
    if not process.stdout.strip():
        raise RuntimeError(f'Native verifier exited {process.returncode}: {process.stderr}')
    report = json.loads(process.stdout)
    if process.returncode != 0 or report.get('passed') is not True:
        raise RuntimeError(f'{group_name}, d={d}, q={q}: {report}')
    assert report['verification_scope'] == 'whole_source'
    assert report['checked_input_count'] == rank
    assert len(report['relations']) == len(spec['relations'])
    for result, relation in zip(report['relations'], spec['relations']):
        assert result['name'] == relation['name']
        assert result['passed'] is True and result['witness_file_reused'] is True
        assert result['verification_route'] in ('ordinary_polynomial', 'compact_witness_replay')
    return report

def replay_nim_worker(case):
    """
    Verify one case and return a short report to the notebook process.
    Each case loads its own source and recorded intermediate elements.
    The defining equations are checked by the native replay, without
    dependencies on other cases or any search for new elements.
    """
    group_name, d, q = case
    report = replay_mod81_case_nim(group_name, d, q)
    return {
        'group': group_name, 'degree': d, 'orientation': q,
        'rank': report['checked_input_count'], 'passed': report['passed'],
    }

def run_parallel_nim_replay(cases, workers):
    """
    Replay the given cases in bounded batches of at most workers processes.

    Submit cases in their supplied order; reports may finish out of order.
    A fresh process pool is used for each batch so that worker memory is
    released before the next batch. No mathematical condition is omitted.

    On a failed case, allow the current batch to finish, then raise an
    exception without submitting another batch. Return the successful
    summaries only if every requested case passes.

    The fork context supports functions defined in this Linux Sage notebook.
    """
    if workers != int(workers) or workers < 1:
        raise ValueError('NIM_WORKERS must be a positive integer')
    workers = int(workers)
    cases = list(cases)
    results = []
    context = get_context('fork')
    for start in range(0, len(cases), workers):
        batch = cases[start:start+workers]
        failures = []
        print('Starting batch:', batch, flush=True)
        with ProcessPoolExecutor(max_workers=len(batch), mp_context=context) as executor:
            pending = {executor.submit(replay_nim_worker, case): case for case in batch}
            for future in as_completed(pending):
                case = pending[future]
                try:
                    result = future.result()
                except Exception as error:
                    failures.append((case, repr(error)))
                    print('ERROR:', case, repr(error), flush=True)
                else:
                    results.append(result)
                    print(result, flush=True)
        if failures:
            raise RuntimeError(f'Replay stopped after failed batch: {failures}')
    return results

if RUN_NIM_REPLAY:
    nim_cases = [
        (group_name, d, q)
        for group_name, group in groups.items()
        for d in group['degrees']
        for q in (0,1)
    ]
    nim_results = run_parallel_nim_replay(nim_cases, NIM_WORKERS)
    print('Cases replayed:', len(nim_results))
    print('ALL REQUESTED FINITE-SOURCE CONDITIONS VERIFIED BY NIM REPLAY')
else:
    print('Optional Nim replay is disabled. Set RUN_NIM_REPLAY = True to run it.')


Starting batch: [('T7_mod81', 0, 0), ('T7_mod81', 0, 1), ('T7_mod81', 2, 0), ('T7_mod81', 2, 1)]
{'group': 'T7_mod81', 'degree': 0, 'orientation': 0, 'rank': 0, 'passed': True}
{'group': 'T7_mod81', 'degree': 0, 'orientation': 1, 'rank': 0, 'passed': True}
{'group': 'T7_mod81', 'degree': 2, 'orientation': 1, 'rank': 0, 'passed': True}
{'group': 'T7_mod81', 'degree': 2, 'orientation': 0, 'rank': 1, 'passed': True}
Starting batch: [('T7_mod81', 4, 0), ('T7_mod81', 4, 1), ('T7_mod81', 6, 0), ('T7_mod81', 6, 1)]
{'group': 'T7_mod81', 'degree': 6, 'orientation': 1, 'rank': 1, 'passed': True}
{'group': 'T7_mod81', 'degree': 4, 'orientation': 0, 'rank': 1, 'passed': True}
{'group': 'T7_mod81', 'degree': 4, 'orientation': 1, 'rank': 0, 'passed': True}
{'group': 'T7_mod81', 'degree': 6, 'orientation': 0, 'rank': 1, 'passed': True}
Starting batch: [('T7_mod81', 8, 0), ('T7_mod81', 8, 1), ('T7_mod81', 10, 0), ('T7_mod81', 10, 1)]
{'group': 'T7_mod81', 'degree': 8, 'orientation': 1, 'rank': 1, 'pa

## Strong realization of the permitted signatures

Check that every permitted signature has a saved strong representative.
Read the manuscript's appendix table and the scan in
`strong_signatures/p3_m4/`, including the exact eigenform records.
Every even weight through 214 is checked; no new scan is run.

Write a signature as $(k\bmod54,a,b,\beta)$, meaning
$a_2\equiv_{\mathrm{val}}a+b\omega$ and
$a_7\equiv_{\mathrm{val}}\beta$ modulo 81, where $\omega^2=2$.
Retain both members of each conjugate pair. There are 147 rational
signatures and 12 nonrational signatures, not merely six extra classes.

The code checks record digests, orbit dimensions and local-place
counts, and identifies a representative (weight, orbit, place) for
each signature. A digest checks consistency of the saved record;
it does not prove an eigenform equation. The exact eigenform
computations and their saved verification results are inputs here.

For a nonrational packet, check the unramified quadratic completion,
the rational $T_7$ residue and the $T_2$ residue modulo 27 by ideal
membership. **Using the forward classification in the manuscript**,
these distinguish the permitted conjugate pairs
$9\pm27\omega$ and $72\pm27\omega$. Frobenius interchanges their
members, so both give strong realizations. This identification is
not an independent proof of the forward classification.

The following cells can run independently of the source replay loops.
They check the finite realization step. The all-weight conclusion
also uses propagation, passage to the target lattices and
completed-Hecke generation from the manuscript.


In [8]:
# Use Sage's exact rationals throughout the ideal-coordinate checks.
# They are compatible with the integers produced by Sage's notebook preparser.
# This cell reads saved records; it does not rerun an eigenform scan.
import hashlib
import json
import re
from sage.all import QQ
from pathlib import Path


def verify_mod81_strong_signatures(repository_root):
    """Compare the recorded strong signatures with the manuscript table, including nonrational entries.

    Check record consistency and the stored ideal-coordinate equations used for
    KRW reduction. For the six nonrational conjugate pairs, use the forward
    classification to identify the two possible values from their rational
    reduction modulo 27. This does not recompute eigenforms or independently
    prove the all-weight classification.
    """
    root = Path(repository_root)
    scan = root / "strong_signatures/p3_m4"
    manuscript = root / "draft/prime_power_congruences_level_one_dickson_twisted.tex"

    def require(condition, message):
        """Reject a failed consistency or arithmetic check with the specified explanation."""
        if not condition:
            raise AssertionError(message)

    def digest(value):
        """Compute the record identifier in the native producer's JSON convention."""
        encoded = json.dumps(value, ensure_ascii=False, separators=(",", ":"))
        return hashlib.sha1(encoded.encode("utf-8")).hexdigest().upper()

    def sealed_record(path):
        """Load a JSON record and check its stored payload digest before using its contents.

        The digest checks file consistency; the arithmetic checks are performed separately.
        """
        record = json.loads(path.read_text())
        payload = {k: v for k, v in record.items() if k != "payload_sha1"}
        require(digest(payload) == record.get("payload_sha1", "").upper(),
                f"Record digest mismatch: {path}")
        return record

    # A signature is (k mod 54, a, b, beta), with alpha = a + b*omega,
    # omega^2 = 2, and all three coefficients reduced modulo 81.
    c_r = {
        0: 1, 4: 4, 6: 4, 10: 1, 12: 7, 16: -2,
        18: 10, 22: -5, 24: 13, 28: -8, 30: 16, 34: 16,
        36: -8, 40: 13, 42: -5, 46: 10, 48: -2, 52: 7,
    }
    expected = set()
    for k in range(0, 54, 2):
        r = (k - 2) % 54
        beta = (1 + pow(7, r + 1, 81)) % 81
        if r in c_r:
            values = [(3*u, 0) for u in range(27)
                      if (u*u - c_r[r]) % 27 in (0, 9)]
        else:
            values = [(a, 0) for a in range(0, 81, 9)
                      if r not in (8, 26, 44) or a not in (9, 72)]
            if r in (8, 26, 44):
                values += [(a, b) for a in (9, 72) for b in (27, 54)]
        expected.update((k, a, b, beta) for a, b in values)
    require(len(expected) == 159, "Unexpected number of permitted signatures")

    # Check the actual appendix, including its definition of the set E.
    text = manuscript.read_text()
    appendix = text.split(r"\label{app:p3-signatures}", 1)[1]
    appendix = appendix.split(r"\label{app:p7-signatures}", 1)[0]

    def alpha_values(entries):
        """Parse the appendix representatives as coefficient pairs in 1 and omega modulo 81."""
        entries = re.sub(r"\s+", "", entries.replace(r"\,", ""))
        values = []
        for entry in entries.split(","):
            pair = re.fullmatch(r"(\d+)\\pm(\d+)\\omega", entry)
            if pair:
                a, b = map(int, pair.groups())
                values.extend([(a % 81, b % 81), (a % 81, -b % 81)])
            else:
                values.append((int(entry) % 81, 0))
        require(len(values) == len(set(values)), "Repeated appendix value")
        return values

    match = re.search(r"\\mathcal E:=\s*\\\{(.*?)\\\}", appendix, re.S)
    require(match is not None, "Missing appendix definition of E")
    exceptional = alpha_values(match.group(1))
    listed = set()
    row_residues = []
    for line in appendix.splitlines():
        match = re.match(r"^(\d+)\s*&\s*\\\((.*?)\\\)\s*&\s*\\\((\d+)\\\)", line)
        if match:
            k, entries, beta = match.groups()
            row_residues.append(int(k))
            values = exceptional if entries == r"\mathcal E" else alpha_values(entries)
            listed.update((int(k), a, b, int(beta)) for a, b in values)
    require(row_residues == list(range(0, 54, 2)), "Missing or repeated appendix row")
    require(listed == expected, "Appendix disagrees with the permitted signatures")

    summary = json.loads((scan / "summary.json").read_text())
    require(summary["prime"] == 3 and summary["exponent"] == 4
            and summary["hecke_indices"] == [2, 7], "Wrong scan parameters")
    weights = list(range(2, 215, 2))
    require(set(weights) <= set(summary["completed_weights"]),
            "The saved scan does not cover every even weight through 214")
    first = {}
    pair_representatives = {}

    for weight in weights:
        record = sealed_record(scan / f"weight_{weight}.json")
        require(record["schema"] == "hecke.strong-signatures.v1"
                and record["weight"] == weight and record["level"] == 1
                and record["cuspidal"] and record["finite_weight_complete"]
                and record["prime"] == 3 and record["exponent"] == 4
                and int(record["modulus"]) == 81
                and record["hecke_indices"] == [2, 7]
                and record["weight_period"] == 54
                and record["weight_residue"] == weight % 54
                and record["degree"] == weight - 2
                and record["degree_residue"] == (weight - 2) % 54,
                f"Invalid weight record: {weight}")
        exact = sealed_record(root / "strong_signatures/exact" / f"weight_{weight}.json")
        require(digest(exact) == record["exact_cache_sha1"].upper(),
                f"Exact eigenform record mismatch: {weight}")
        require(exact["weight"] == weight and exact["level"] == 1
                and exact["cuspidal"] and exact["orbit_dimension_sum_verified"]
                and exact["dimension"] == record["dimension"],
                f"Invalid exact cache: {weight}")
        require([(o["orbit"], o["field_degree"]) for o in exact["orbits"]]
                == [(o["orbit"], o["field_degree"]) for o in record["orbits"]]
                and all(o["exact_eigenvector_checks_passed"] for o in exact["orbits"]),
                f"Exact eigenform orbit mismatch: {weight}")
        require(sum(o["field_degree"] for o in record["orbits"]) == record["dimension"],
                f"Incomplete eigenform orbits: {weight}")

        k = weight % 54
        r = (weight - 2) % 54
        beta = (1 + pow(7, r + 1, 81)) % 81
        for orbit in record["orbits"]:
            embedding_count = 0
            for packet in orbit["local_packets"]:
                e, f = packet["ramification_index"], packet["residue_degree"]
                require(packet["local_embedding_count"] == e*f
                        and packet["krw_ideal_exponent"] == 3*e + 1
                        and packet["krw_reduction_replayed"],
                        f"Invalid local packet: weight {weight}")
                embedding_count += e*f
                representative = (weight, orbit["orbit"], packet["place"])
                rational = packet["rational_signature"]
                if rational is not None:
                    require(len(rational) == 2, "Expected two Hecke coordinates")
                    a, t = map(int, rational)
                    signatures = [(k, a, 0, t)]
                else:
                    require((e, f) == (1, 2) and r in (8, 26, 44),
                            f"Unexpected nonrational packet: {representative}")
                    columns = packet["quotient_ideal_hnf_columns"]
                    n = len(columns)
                    require(n > 0 and all(len(col) == n for col in columns),
                            "Invalid ideal HNF dimensions")
                    H = [[QQ(columns[j][i]) for j in range(n)] for i in range(n)]
                    require(all(H[i][i] > 0 for i in range(n))
                            and all(H[i][j] == 0 for i in range(n) for j in range(i))
                            and packet["p_maximal_basis"][0] == "1",
                            "Invalid ideal HNF or basis")
                    determinant = QQ(1)
                    for i in range(n):
                        determinant *= H[i][i]
                    require(determinant == 3**(packet["krw_ideal_exponent"]*f),
                            "Wrong KRW ideal norm")
                    a, t = [list(map(QQ, row))
                            for row in packet["eigenvalue_residue_basis_coordinates"]]
                    require(len(a) == len(t) == n, "Wrong coordinate dimensions")

                    def in_ideal(vector):
                        """Test membership in the localized ideal lattice using its column Hermite basis.

                        Denominators prime to 3 are units; no full maximal-order calculation is used here.
                        """
                        # H is a column HNF. Solve H*x=vector over Q, then
                        # test x in Z_(3)^n; prime-to-3 denominators are units.
                        x = [QQ(0)]*n
                        for i in reversed(range(n)):
                            x[i] = (vector[i] - sum(H[i][j]*x[j]
                                                   for j in range(i+1, n))) / H[i][i]
                        return all(coefficient.denominator() % 3 != 0 for coefficient in x)

                    def difference(vector, scalar, multiplier=1):
                        """Form the scaled coordinate difference from the scalar multiple of the unit element."""
                        return [multiplier*(value - (scalar if i == 0 else 0))
                                for i, value in enumerate(vector)]

                    require(in_ideal(difference(t, beta)), "Wrong T7 residue")
                    require(not any(in_ideal(difference(a, scalar)) for scalar in range(81)),
                            "The purported nonrational T2 residue is rational")
                    # e=1, so multiplication by 3 tests equality modulo P^3
                    # using the stored ideal P^4 (literal mod 81).
                    residues = [scalar for scalar in range(27)
                                if in_ideal(difference(a, scalar, 3))]
                    require(len(residues) == 1 and residues[0] in (9, 18),
                            "Unexpected T2 residue modulo 27")
                    s = residues[0] // 9
                    # By the forward classification, these are the only two
                    # nonrational possibilities with this r and s. Frobenius
                    # on the unramified quadratic completion exchanges them.
                    base = 9 if s == 1 else 72
                    signatures = [(k, base, b, beta) for b in (27, 54)]
                    pair_representatives.setdefault((r, s), representative)
                for signature in signatures:
                    require(signature in expected,
                            f"Unexpected signature {signature} at {representative}")
                    first.setdefault(signature, representative)
            require(embedding_count == orbit["field_degree"],
                    f"Incomplete local places: weight {weight}, orbit {orbit['orbit']}")

    require(set(first) == expected, f"Unrealized signatures: {sorted(expected - set(first))}")
    require(set(pair_representatives) == {(r, s) for r in (8, 26, 44) for s in (1, 2)},
            "A nonrational conjugate pair is missing")
    rational_count = sum(b == 0 for k, a, b, beta in first)
    bound = max(reference[0] for reference in first.values())
    require(rational_count == 147 and bound == 214, "Unexpected count or weight bound")
    return {
        "weight_files_audited": len(weights), "exact_caches_audited": len(weights),
        "rational_signatures": rational_count, "nonrational_signatures": len(first)-rational_count,
        "strong_signatures": len(first), "realization_weight_bound": bound,
        "nonrational_pair_representatives": pair_representatives,
        "representatives": first, "finite_realization_verified": True,
        "all_weight_classification_proved_by_this_cell": False,
    }


In [9]:
strong_mod81_verification = verify_mod81_strong_signatures(Path.cwd())
print("Appendix signature table and saved eigenform records agree.")
print("Weight files / exact caches audited:", strong_mod81_verification["weight_files_audited"])
print("Rational signatures:", strong_mod81_verification["rational_signatures"])
print("Nonrational signatures:", strong_mod81_verification["nonrational_signatures"])
print("Total strong signatures:", strong_mod81_verification["strong_signatures"])
print("Realization weight bound:", strong_mod81_verification["realization_weight_bound"])
print("Nonrational pairs: (r,s) -> (weight,orbit,place)")
for pair, representative in sorted(strong_mod81_verification["nonrational_pair_representatives"].items()):
    print(pair, "->", representative)
print("ALL 159 PERMITTED SIGNATURES HAVE STRONG REALIZATIONS BY WEIGHT 214")


Appendix signature table and saved eigenform records agree.
Weight files / exact caches audited: 107
Rational signatures: 147
Nonrational signatures: 12
Total strong signatures: 159
Realization weight bound: 214
Nonrational pairs: (r,s) -> (weight,orbit,place)
(8, 1) -> (64, 1, 4)
(8, 2) -> (118, 1, 7)
(26, 1) -> (136, 1, 7)
(26, 2) -> (28, 1, 1)
(44, 1) -> (100, 1, 6)
(44, 2) -> (46, 1, 2)
ALL 159 PERMITTED SIGNATURES HAVE STRONG REALIZATIONS BY WEIGHT 214
